In [1]:
import re
import json
import os
import mwparserfromhell

In [2]:
def parse_rezept_template(wikitext):
    parsed = mwparserfromhell.parse(wikitext)
    templates = parsed.filter_templates()
    
    metadata = {}
    
    for template in templates:
        name = str(template.name).strip().lower()
        
        if name == "rezept":
            for param in template.params:
                key = str(param.name).strip()
                value = str(param.value).strip()
                
                if key and value:
                    metadata[key.lower()] = value
    
    return metadata


def extract_sections(wikitext):
    sections = {}
    current_section = "intro"
    current_content = []
    
    for line in wikitext.split("\n"):
        header_match = re.match(r'^(={2,3})\s*(.+?)\s*\1\s*$', line)
        
        if header_match:
            if current_content:
                sections[current_section] = "\n".join(current_content).strip()
            
            current_section = header_match.group(2).strip()
            current_content = []
        else:
            current_content.append(line)
    
    if current_content:
        sections[current_section] = "\n".join(current_content).strip()
    
    return sections


def parse_zutaten(zutaten_text):
    zutaten = []
    
    for line in zutaten_text.split("\n"):
        line = line.strip()
        
        if not line.startswith("*"):
            continue
        
        line = line.lstrip("* ").strip()
        
        if not line:
            continue
        
        line_clean = re.sub(r'\[\[(?:Zutat:)?([^|\]]*\|)?([^\]]+)\]\]', r'\2', line)
        line_clean = re.sub(r"'{2,3}", "", line_clean)  
        line_clean = line_clean.strip()
        
        zutat_parsed = parse_single_zutat(line_clean)
        zutat_parsed["raw"] = line_clean
        zutaten.append(zutat_parsed)
    
    return zutaten


def parse_single_zutat(text):
    einheiten = [
        "kg", "g", "mg", "l", "ml", "cl", "dl",
        "EL", "TL", "Tasse", "Tassen", "Becher",
        "Prise", "Prisen", "Bund", "Stück", "Scheibe", "Scheiben",
        "Packung", "Päckchen", "Dose", "Dosen", "Glas", "Gläser",
        "Blatt", "Blätter", "Zweig", "Zweige", "Zehe", "Zehen",
        "Messerspitze", "Msp", "Handvoll",
    ]
    
    pattern = r'^(\d+[\d.,/–\-\s]*(?:\d+)?)\s*(' + '|'.join(einheiten) + r')?\s*(.+)$'
    match = re.match(pattern, text, re.IGNORECASE)
    
    if match:
        return {
            "menge": match.group(1).strip(),
            "einheit": (match.group(2) or "").strip(),
            "name": match.group(3).strip()
        }
    
    vague_pattern = r'^(etwas|wenig|viel|nach Bedarf|nach Geschmack|evtl\.?)\s+(.+)$'
    vague_match = re.match(vague_pattern, text, re.IGNORECASE)
    
    if vague_match:
        return {
            "menge": vague_match.group(1).strip(),
            "einheit": "",
            "name": vague_match.group(2).strip()
        }
    
    return {
        "menge": "",
        "einheit": "",
        "name": text
    }


def wikitext_to_plaintext(wikitext):
    parsed = mwparserfromhell.parse(wikitext)
    
    for template in parsed.filter_templates():
        try:
            parsed.remove(template)
        except ValueError:
            pass
    
    text = str(parsed)
    
    text = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]+)\]\]', r'\1', text)
    
    text = re.sub(r'\[https?://[^\s\]]+ ([^\]]+)\]', r'\1', text)
    text = re.sub(r'\[https?://[^\]]+\]', '', text)
    
    text = re.sub(r"'{2,5}", "", text)  # '''fett''', ''kursiv''
    
    text = re.sub(r'<ref[^>]*>.*?</ref>', '', text, flags=re.DOTALL)
    text = re.sub(r'<[^>]+/?>', '', text)
    
    text = re.sub(r'^(={2,4})\s*(.+?)\s*\1\s*$', r'\2', text, flags=re.MULTILINE)

    text = re.sub(r'^\*+\s*', '• ', text, flags=re.MULTILINE)
    text = re.sub(r'^#+\s*', '', text, flags=re.MULTILINE)
    
    text = re.sub(r'\[\[Kategorie:[^\]]+\]\]', '', text)

    text = re.sub(r'\{\|.*?\|\}', '', text, flags=re.DOTALL)
    
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    text = re.sub(r'__[A-Z]+__', '', text)
    
    return text.strip()


def parse_recipe(wikitext, title="", categories=None):
    if categories is None:
        categories = []
    
    metadata = parse_rezept_template(wikitext)
    
    sections = extract_sections(wikitext)
    
    zutaten_raw = ""
    for key in sections:
        if "zutat" in key.lower() or "ingredient" in key.lower():
            zutaten_raw = sections[key]
            break
    
    zutaten = parse_zutaten(zutaten_raw) if zutaten_raw else []
    
    zubereitung = ""
    for key in sections:
        if "zubereitung" in key.lower() or "anleitung" in key.lower():
            zubereitung = sections[key]
            break
    
    plaintext = wikitext_to_plaintext(wikitext)
    
    kueche = [cat for cat in categories if "Küche" in cat or "küche" in cat]
    
    schwierigkeit = metadata.get("schwierigkeit", "").lower()
    
    result = {
        "title": title,
        "metadata": {
            "menge": metadata.get("menge", ""),
            "zeit": metadata.get("zeit", ""),
            "schwierigkeit": schwierigkeit,
            "alkohol": metadata.get("alkohol", ""),
            "bild": metadata.get("bild", ""),
        },
        "categories": categories,
        "kueche": kueche,
        "zutaten": zutaten,
        "zutaten_namen": [z["name"] for z in zutaten],  
        "zubereitung_raw": wikitext_to_plaintext(zubereitung) if zubereitung else "",
        "sections": {k: wikitext_to_plaintext(v) for k, v in sections.items()},
        "plaintext": plaintext,  
        "wikitext": wikitext,   
    }
    
    return result


def parse_zutat_template(wikitext):
    parsed = mwparserfromhell.parse(wikitext)
    templates = parsed.filter_templates()
    
    metadata = {}
    
    for template in templates:
        name = str(template.name).strip().lower()
        
        if name in ("zutat", "zutatübersicht"):
            metadata["template_type"] = name
            for param in template.params:
                key = str(param.name).strip()
                value = str(param.value).strip()
                if key and value:
                    metadata[key.lower()] = value
    
    return metadata


def parse_zutat(wikitext, title="", categories=None):
    if categories is None:
        categories = []
    
    clean_name = title.replace("Zutat:", "").strip()
    
    template_data = parse_zutat_template(wikitext)
    
    naehrwerte = {}
    naehrwert_keys = {
        "kcal": "kcal", "kj": "kj",
        "fett": "fett", "kohlenhydrate": "kohlenhydrate",
        "eiweiß": "eiweiss", "cholesterin": "cholesterin",
        "ballaststoffe": "ballaststoffe",
    }
    
    for raw_key, clean_key in naehrwert_keys.items():
        value = template_data.get(raw_key, "")
        num_match = re.search(r'([\d.,]+)', value)
        if num_match:
            try:
                naehrwerte[clean_key] = float(num_match.group(1).replace(",", "."))
            except ValueError:
                naehrwerte[clean_key] = None
        else:
            naehrwerte[clean_key] = None
    
    basismenge = template_data.get("basismenge", "")
    
    sections = extract_sections(wikitext)
    
    plaintext = wikitext_to_plaintext(wikitext)
    
    verwandte_zutaten = re.findall(r'\[\[Zutat:([^|\]]+)', wikitext)
    verwandte_zutaten = list(set(verwandte_zutaten)) 
    
    result = {
        "type": "zutat",
        "title": title,
        "name": clean_name,
        "bild": template_data.get("bild", ""),
        "basismenge": basismenge,
        "naehrwerte": naehrwerte,
        "has_naehrwerte": any(v is not None for v in naehrwerte.values()),
        "categories": categories,
        "verwandte_zutaten": verwandte_zutaten,
        "sections": {k: wikitext_to_plaintext(v) for k, v in sections.items()},
        "plaintext": plaintext,
        "wikitext": wikitext,
    }
    
    return result


def parse_all_recipes(input_file, output_file):
 
    print(f"Lade {input_file}...")
    with open(input_file, "r", encoding="utf-8") as f:
        raw_recipes = json.load(f)
    
    print(f"{len(raw_recipes)} recipes loaded. Start Parsing...")
    
    parsed_recipes = []
    errors = []
    
    for i, recipe in enumerate(raw_recipes):
        try:
            parsed = parse_recipe(
                wikitext=recipe["wikitext"],
                title=recipe["title"],
                categories=recipe.get("categories", [])
            )
            parsed_recipes.append(parsed)
            
            if (i + 1) % 500 == 0:
                print(f"  [{i+1}/{len(raw_recipes)}] processed...")
                
        except Exception as e:
            errors.append({"title": recipe.get("title", "?"), "error": str(e)})
    
    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(parsed_recipes, f, ensure_ascii=False, indent=2)
    
    print(f"\n✓ {len(parsed_recipes)} recipes successfully parsed")
    print(f"✗ {len(errors)} Errors")
    print(f"→ Saved to {output_file}")
    
    if errors:
        error_file = output_file.replace(".json", "_errors.json")
        with open(error_file, "w", encoding="utf-8") as f:
            json.dump(errors, f, ensure_ascii=False, indent=2)
        print(f"→ Fehler-Log in {error_file}")
    
    return parsed_recipes


def parse_all_zutaten(input_file, output_file):
    print(f"Load {input_file}...")
    with open(input_file, "r", encoding="utf-8") as f:
        raw_zutaten = json.load(f)
    
    print(f"{len(raw_zutaten)} ingredients loaded. Start Parsing...")
    
    parsed_zutaten = []
    errors = []
    
    for i, zutat in enumerate(raw_zutaten):
        try:
            parsed = parse_zutat(
                wikitext=zutat["wikitext"],
                title=zutat["title"],
                categories=zutat.get("categories", [])
            )
            parsed_zutaten.append(parsed)
            
            if (i + 1) % 200 == 0:
                print(f"  [{i+1}/{len(raw_zutaten)}] processed...")
                
        except Exception as e:
            errors.append({"title": zutat.get("title", "?"), "error": str(e)})
    
    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(parsed_zutaten, f, ensure_ascii=False, indent=2)
    
    print(f"\n✓ {len(parsed_zutaten)} ingredients successfully parsed")
    print(f"✗ {len(errors)} Errors")
    print(f"→ Saved to {output_file}")
    
    if errors:
        error_file = output_file.replace(".json", "_errors.json")
        with open(error_file, "w", encoding="utf-8") as f:
            json.dump(errors, f, ensure_ascii=False, indent=2)
        print(f"→ Error-Log in {error_file}")
    
    return parsed_zutaten

In [3]:
test_wikitext = """{{Rezept|
 | Menge         = 4 Personen
 | Zeit          = 30–40 Minuten
 | Schwierigkeit = leicht
 | Alkohol       = nein
 | Bild          = Kein_Bild.png
|}}
Die '''"Krömpele"-Suppe''' ist eine typische thüringische Suppe. ''Krömpele'' ist mundartlich und steht für ''Krümel'', ''Streusel'', ''Brösel''.

== Zutaten ==
* 1 l [[Zutat:Fleischbrühe|Fleischbrühe]]
* 250 g [[Zutat:Weißbrot|Weißbrot]], altbacken
* 2 [[Zutat:Ei|Eier]]
* etwas [[Zutat:Salz|Salz]]
* 1 Prise [[Zutat:Muskatnuss|Muskatnuss]]
* 2 EL [[Zutat:Schnittlauch|Schnittlauch]], fein geschnitten

== Zubereitung ==
# Das [[Zutat:Weißbrot|Weißbrot]] in kleine Würfel schneiden.
# Die [[Zutat:Ei|Eier]] verquirlen und mit [[Zutat:Salz|Salz]] und [[Zutat:Muskatnuss|Muskatnuss]] würzen.
# Die Brotwürfel in der Eiermasse wenden.
# Die [[Zutat:Fleischbrühe|Fleischbrühe]] zum Kochen bringen.
# Die Brot-Ei-Masse mit einem Löffel als kleine Nocken in die kochende Brühe geben.
# 5–10 Minuten ziehen lassen.
# Mit [[Zutat:Schnittlauch|Schnittlauch]] bestreut servieren.

== Beilagen ==
* Frisches Brot

[[Kategorie:Brotsuppen]]
[[Kategorie:Thüringer Küche]]
"""
    
print("=== KOCHWIKI PARSER - Quick Test ===\n")
print("─" * 50)
print("TEST 1: REZEPT")
print("─" * 50)
    
result = parse_recipe(
    test_wikitext, 
    title='"Krömpele"-Suppe',
    categories=["Brotsuppen", "Nocken", "Rezepte ohne Alkohol", 
                "Schwierigkeit leicht", "Thüringer Küche"]
    )
    
print(f"Titel: {result['title']}")
print(f"Metadata: {json.dumps(result['metadata'], ensure_ascii=False, indent=2)}")
print(f"Küche: {result['kueche']}")
print(f"Anzahl Zutaten: {len(result['zutaten'])}")
print(f"\nZutaten:")
for z in result["zutaten"]:
    print(f"  {z['menge']:>8} {z['einheit']:<6} {z['name']}")
    
print(f"\nZubereitung:")
print(f"  {result['zubereitung_raw'][:300]}...")
    

test_zutat_butter = """{{Zutat|
 | Bild          = Echire.jpg
 | Basismenge    = 100 g Butter
 | kj            = 3157 kJ
 | kcal          = 754 kcal
 | Fett          = 83,2 g
 | Kohlenhydrate = 0,6 g
 | Eiweiß        = 0,7 g
 | Cholesterin  = 240 mg
 | Ballaststoffe = 0,0 g
|}}
'''Butter''', oder, mit den Worten eines irischen Poeten, geronnenes Sonnenlicht, ist ein aus [[Zutat:Milch|Kuhmilch]] gewonnenes Streichfett, das zu 82 % aus Milchfett, zu 16 % aus Wasser und zu 2 % aus verschiedenen anderen Stoffen besteht.

== Sorten ==
* Deutsche Markenbutter
* Sauerrahmbutter
* Süßrahmbutter

== Lagerung ==
Butter sollte kühl und dunkel gelagert werden.
"""
    
print(f"\n{'─' * 50}")
print("TEST 2: ZUTAT (mit Nährwerten)")
print("─" * 50)
    
zutat_result = parse_zutat(
    test_zutat_butter,
    title="Zutat:Butter",
    categories=["Tierische Fette", "Zutaten"]
    )
    
print(f"Name: {zutat_result['name']}")
print(f"Basismenge: {zutat_result['basismenge']}")
print(f"Nährwerte: {json.dumps(zutat_result['naehrwerte'], indent=2)}")
print(f"Verwandte Zutaten: {zutat_result['verwandte_zutaten']}")
print(f"Kategorien: {zutat_result['categories']}")
print(f"Sections: {list(zutat_result['sections'].keys())}")
    

test_zutat_mehl = """{{Zutatübersicht|
 | Bild          = Flours.jpg
|}}
'''Mehl''' ist gemahlenes [[Zutat:Getreide|Getreide]] und wird als Basis von Teigen zur Herstellung von Backwaren oder Teigwaren verwendet. Wird in Rezepten nur von „Mehl" gesprochen, ist für gewöhnlich [[Zutat:Weizenmehl|Weizenmehl]] der Type 405 gemeint.

== Typisierung nach DIN ==
Die Ermittlung der Type bzw. Helligkeit erfolgt durch die Bestimmung des Mineralstoffgehaltes.
"""
    
print(f"\n{'─' * 50}")
print("TEST 3: ZUTAT (ohne Nährwerte / Zutatübersicht)")
print("─" * 50)
    
zutat_result2 = parse_zutat(
    test_zutat_mehl,
    title="Zutat:Mehl",
    categories=["Zutaten"]
    )
    
print(f"Name: {zutat_result2['name']}")
print(f"Hat Nährwerte: {zutat_result2['has_naehrwerte']}")
print(f"Verwandte Zutaten: {zutat_result2['verwandte_zutaten']}")
print(f"Sections: {list(zutat_result2['sections'].keys())}")
print(f"\nPlaintext (erste 300 Zeichen):")
print(f"  {zutat_result2['plaintext'][:300]}...")

=== KOCHWIKI PARSER - Quick Test ===

──────────────────────────────────────────────────
TEST 1: REZEPT
──────────────────────────────────────────────────
Titel: "Krömpele"-Suppe
Metadata: {
  "menge": "4 Personen",
  "zeit": "30–40 Minuten",
  "schwierigkeit": "leicht",
  "alkohol": "nein",
  "bild": "Kein_Bild.png"
}
Küche: ['Thüringer Küche']
Anzahl Zutaten: 6

Zutaten:
         1 l      Fleischbrühe
       250 g      Weißbrot, altbacken
         2        Eier
     etwas        Salz
         1 Prise  Muskatnuss
         2 EL     Schnittlauch, fein geschnitten

Zubereitung:
  Das Weißbrot in kleine Würfel schneiden.
Die Eier verquirlen und mit Salz und Muskatnuss würzen.
Die Brotwürfel in der Eiermasse wenden.
Die Fleischbrühe zum Kochen bringen.
Die Brot-Ei-Masse mit einem Löffel als kleine Nocken in die kochende Brühe geben.
5–10 Minuten ziehen lassen.
Mit Schnittlauch...

──────────────────────────────────────────────────
TEST 2: ZUTAT (mit Nährwerten)
────────────────────────────